# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Rodrigo Enzo Hernández**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

# Algoritmo elegido

**Búsqueda A\* usando la heurística dada en el aula virtual**

# 1. Importar librerías

In [141]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue

# 2. Funciónes nuevas

## 2.1 Función para crear el diccionario de métricas

In [142]:
def create_metrics(solution_found: bool, nodes_explored: int, states_visited: set, 
                   nodes_in_frontier: int, max_depth: int, cost_total: float) -> dict:
    """
    Función auxiliar para crear el diccionario de métricas.
    
    Args:
        solution_found: True si se encontró solución
        nodes_explored: Número de nodos explorados
        states_visited: Conjunto de estados visitados
        nodes_in_frontier: Número de nodos en la frontera
        max_depth: Profundidad máxima alcanzada
        cost_total: Costo total del camino
    
    Returns:
        dict: Diccionario con las métricas
    """
    return {
        "solution_found": solution_found,
        "nodes_explored": nodes_explored,
        "states_visited": len(states_visited),
        "nodes_in_frontier": nodes_in_frontier,
        "max_depth": max_depth,
        "cost_total": cost_total,
    }

## 2.2 Función heurística para la Torre de Hanoi

In [143]:
def heuristic_hanoi(node: NodeHanoi, problem: ProblemHanoi) -> int:
    """
    Función heurística para la Torre de Hanoi.
    
    EXPLICACIÓN:
    - Una heurística es una "estimación" de qué tan lejos estamos del objetivo
    - En este caso, contamos cuántos discos ya están en la posición correcta (varilla objetivo)
    - Luego, devolvemos el NEGATIVO de ese número: h(n) = -k
    
    ¿Por qué negativo?
    - Queremos minimizar el valor de f(n) = g(n) + h(n)
    - Si más discos están en su lugar correcto, h(n) es más pequeño
    - Esto hace que A* priorice estados más cercanos al objetivo
    
    Args:
        node: El nodo actual que estamos evaluando
        problem: El problema de Hanoi (contiene el estado objetivo)
    
    Returns:
        int: El valor heurístico h(n) = -k (negativo de discos correctos)
    """
    current_state = node.state
    goal_state = problem.goal
    
    # Contamos cuántos discos están en la varilla objetivo
    correctly_placed_disks = len(set(current_state.rods[2]) & set(goal_state.rods[2]))
    
    return -correctly_placed_disks

# 3. Implementación del algoritmo A*

In [144]:
def astar_search_hanoi(problem: ProblemHanoi):
    """
    Implementación de búsqueda A* para la Torre de Hanoi.
    
    EXPLICACIÓN DE A*:
    - A* es un algoritmo de búsqueda INFORMADA (usa conocimiento del problema)
    - Combina:
      * g(n): costo real desde el inicio hasta el nodo n
      * h(n): estimación (heurística) del costo desde n hasta el objetivo
    - Fórmula: f(n) = g(n) + h(n)
    - A* siempre explora el nodo con el menor f(n)
    
    EXPLICACIÓN DE GRAPH SEARCH:
    - "Graph search" significa que recordamos qué estados ya visitamos
    - Esto evita ciclos infinitos (volver a explorar el mismo estado)
    
    Args:
        problem: El problema de Hanoi a resolver
    
    Returns:
        tuple: (nodo_solución, métricas_de_rendimiento)
    """
    # MÉTRICAS: Variables para rastrear el rendimiento del algoritmo
    nodes_explored = 0        # Cuántos nodos sacamos de la frontera para explorar
    states_visited = set()    # Set de estados únicos visitados
    max_depth = 0             # La profundidad máxima que alcanzamos en el árbol
    
    # PASO 1: Crear el nodo inicial
    # node = nodo que representa el estado inicial del problema
    node = NodeHanoi(problem.initial)
    
    # VERIFICACIÓN: ¿Ya estamos en el objetivo?
    if problem.goal_test(node.state):
        return node, create_metrics(True, nodes_explored, states_visited, 0, node.depth, node.path_cost)
    
    # PASO 2: Crear la frontera
    # La frontera es una cola de prioridad que ordena nodos por f(n) = g(n) + h(n)
    # - 'min': queremos el nodo con el menor f(n)
    # - lambda n: ...: función que calcula f(n) para cada nodo n
    frontier = PriorityQueue('min', f=lambda n: n.path_cost + heuristic_hanoi(n, problem))
    frontier.append(node)
    
    # PASO 3: Crear el conjunto de estados explorados sin duplicados
    explored = set()
    
    # PASO 4: bucle mientras haya nodos en la frontera
    while frontier:
        # PASO 4.1: Sacar el nodo con el menor f(n) de la frontera
        _, node = frontier.pop()
        nodes_explored += 1
        
        # Actualizar la profundidad máxima
        if node.depth > max_depth:
            max_depth = node.depth
        
        # PASO 4.2: ¿Es este el objetivo?
        if problem.goal_test(node.state):
            # Encontramos la solución
            return node, create_metrics(True, nodes_explored, states_visited, len(frontier), max_depth, node.path_cost)
        
        # PASO 4.3: Marcar este estado como explorado
        explored.add(node.state)
        states_visited.add(node.state)
        
        # PASO 4.4: expandir el nodo (generar todos los hijos)
        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                # Estado nuevo: agregar a la frontera
                frontier.append(child)
                states_visited.add(child.state)
            
            # Si el hijo ya está en la frontera
            elif child in frontier:
                # Obtenemos el f(n) del nodo que ya está en la frontera
                existing_f = frontier[child]
                
                # Calculamos f(n) del nuevo camino al mismo estado
                new_f = child.path_cost + heuristic_hanoi(child, problem)
                
                # ¿El nuevo camino es mejor?
                if new_f < existing_f:
                    # Sí: reemplazamos el nodo viejo con el nuevo
                    del frontier[child]
                    frontier.append(child)
    
    # PASO 5: Si salimos del bucle, no encontramos solución
    return None, create_metrics(False, nodes_explored, states_visited, 0, max_depth, None)

# 4. Punto de entrada

In [145]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    """
    Función principal que configura y ejecuta el algoritmo de búsqueda.
    
    EXPLICACIÓN:
    - Esta función es el "punto de entrada" del ejercicio
    - Configura el problema (estado inicial y objetivo)
    - Llama al algoritmo A* para resolverlo
    - Devuelve la solución y las métricas
    
    Args:
        number_disks: Número de discos en la Torre de Hanoi (por defecto 5)
    
    Returns:
        tuple: (nodo_solución, diccionario_de_métricas)
    """
    
    # PASO 1: Crear la lista de discos
    # range(5, 0, -1) genera: 5, 4, 3, 2, 1
    # Es decir, del disco más grande (5) al más pequeño (1)
    list_disks = [i for i in range(number_disks, 0, -1)]
    
    # PASO 2: Definir el estado inicial
    # - Varilla 1 (rod1): tiene TODOS los discos [5, 4, 3, 2, 1]
    # - Varilla 2 (rod2): vacía []
    # - Varilla 3 (rod3): vacía []
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    
    # PASO 3: Definir el estado objetivo
    # - Varilla 1 (rod1): vacía []
    # - Varilla 2 (rod2): vacía []
    # - Varilla 3 (rod3): tiene TODOS los discos [5, 4, 3, 2, 1]
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    
    # PASO 4: Crear el problema
    # ProblemHanoi es una clase que encapsula:
    # - El estado inicial
    # - El estado objetivo
    # - Las acciones posibles
    # - La función de transición (qué pasa cuando aplicamos una acción)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)
    
    # PASO 5: Ejecutar A* con la heurística
    # astar_search_hanoi devuelve: (nodo_solución, métricas)
    solution, metrics = astar_search_hanoi(problem)
    
    # PASO 6: Devolver el resultado
    return solution, metrics

# 5. Ejecución y análisis de resultados

Se prueba la función:

In [146]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [147]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 178
states_visited: 193
nodes_in_frontier: 15
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [148]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [149]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
